# Quando transferir um modelo clínico entre hospitaisCódigo de apoio ao trabalho *Quando transferir um modelo clínico entrehospitais: uma lei de decisão validada em 15 unidades de pronto socorro*.**Autor:** Victor Hugo Ovani Marchetti**Orientação:** Prof. Dr. Alexandre Dias Porto Chiavegatto Filho (LabDaps/FSP-USP)**Coorientação:** Dr. José Arnaldo Shiomi da Cruz**Aprovação ética:** CEP Hospital IGESP, parecer 8.426.290, CAAE 96939326.0.0000.5450---## Sobre os dados**Os dados não são distribuídos.** Trata-se de registros assistenciais depacientes, categoria de dado pessoal sensível sob o art. 11 da Lei Geral deProteção de Dados. As consultas de extração ao repositório institucional tambémnão são publicadas.O que este repositório contém é o **método, do arquivo tabular em diante**: comoa coorte é validada, como as partições são construídas para evitar vazamento,como as estratégias de transferência são comparadas e como as métricas sãocalculadas. As unidades aparecem anonimizadas (A, B, C, ...).Para reproduzir com dados próprios, basta um arquivo tabular com o contratodescrito na seção 1.

## 1. Contrato de dadosUma linha por alta de pronto socorro. Nenhuma variável posterior ao momento daalta pode entrar: o modelo é usado *no instante em que o paciente é liberado*.| coluna | tipo | descrição ||---|---|---|| `CD_UNIDADE` | texto | identificador da unidade || `ANO`, `MES` | inteiro, `AAAA-MM` | competência da alta || `ID_PRONTUARIO` | inteiro | identificador do paciente (para evitar vazamento entre partições) || `IDADE` | inteiro | anos completos || `CD_SEXO` | texto | M ou F || `HORAS_PS` | real | permanência no pronto socorro || `HORA_CHEGADA` | inteiro | 0 a 23 || `DIA_SEMANA` | inteiro | 1 a 7 || `CD_CLINICA`, `CD_MOTIVO_ATENDIMENTO`, `CD_PROCEDENCIA` | texto | categóricas do episódio || `N_PS_30D`, `N_PS_90D` | inteiro | passagens prévias pelo PS || `DIAS_ULT_PS` | real | dias desde a última passagem || `N_INT_365D` | inteiro | internações no último ano || `Y` | 0/1 | desfecho: retorno com internação (4 h a 7 d) **ou** óbito (0 a 7 d) |

In [ ]:
import numpy as np, pandas as pd, warningsfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import roc_auc_score, average_precision_score, brier_score_losswarnings.simplefilter("ignore")NUM = ["IDADE", "HORAS_PS", "HORA_CHEGADA", "N_PS_30D", "N_PS_90D",       "DIAS_ULT_PS", "N_INT_365D"]CAT = ["CD_SEXO", "CD_CLINICA", "CD_MOTIVO_ATENDIMENTO", "CD_PROCEDENCIA",       "DIA_SEMANA"]FEATS = NUM + CATANO_TR, ANO_TE = 2024, 2025

## 2. Validação do desfechoTrês afirmações da definição não podem ser presumidas. Cada uma foi medida antesde qualquer modelagem, e as duas primeiras alteram o rótulo.**Continuidade de episódio.** Alta administrativa seguida de internação minutosdepois é o mesmo episódio, com troca de ficha. A janela começa em **4 h**; semessa margem, 18.834 atendimentos entrariam indevidamente como evento.**Componente óbito.** Acrescenta apenas **0,15%** dos eventos: 99,3% dos óbitosjá são precedidos de internação que aciona o desfecho antes.**Retorno, e não qualquer internação.** Exigindo que a passagem pelo PS preceda ainternação, **99,14%** dos eventos são retorno de fato, com mediana de zero horaentre retorno e internação.> **Limitação que permanece.** Óbito fora da rede é invisível. Para alta de PS> esse é o desfecho mais grave, e nenhum resultado aqui o cobre.

## 3. Limpeza: a permanência inverte de sinalA taxa de eventos cresce com a permanência até ~48 h e depois **decresce**,ficando abaixo da média acima de 168 h. Permanência longa verdadeira seriaindicador de gravidade, não de segurança: o padrão revela fichas nuncaencerradas administrativamente.

In [ ]:
def carrega(caminho):    d = pd.read_csv(caminho)    d = d[d.Y.notna()].copy()    d["Y"] = d.Y.astype(int)    for c in NUM:        d[c] = pd.to_numeric(d[c], errors="coerce")    # o decimal precisa ser ponto; vírgula produz coluna toda nula    assert d.IDADE.notna().mean() > .5, "decimais com vírgula?"    d = d[d.IDADE.between(0, 115) & d.HORAS_PS.between(0, 168)].copy()    d["HORAS_PS"] = d.HORAS_PS.clip(upper=72)   # fichas nunca encerradas    return d.dropna(subset=["ANO"])

## 4. Partições sem vazamentoO corte externo é **temporal**: treino em 2024, teste em 2025. Nenhuma decisãoconsulta o ano de teste.O corte interno também é temporal, e por motivo específico: **o mesmo pacientereaparece na coorte**. O retorno ao PS é comum, e é por isso que o número depassagens prévias figura entre os preditores. Validação cruzada que sorteasselinhas colocaria o mesmo prontuário em treino e validação ao mesmo tempo,tornando a seleção de hiperparâmetros otimista. O resíduo de fronteira éremovido por prontuário.

In [ ]:
def parte_temporal(d, frac=.65):    """Passado treina; futuro recalibra. Nunca sorteio."""    meses = np.sort(d.MES.astype(str).unique())    if len(meses) < 4:        return None, None    corte = meses[max(1, int(round(len(meses) * frac))) - 1]    tr, cal = d[d.MES.astype(str) <= corte], d[d.MES.astype(str) > corte]    if len(tr) < 200 or len(cal) < 100 or tr.Y.nunique() < 2:        return None, None    tr = tr[~tr.ID_PRONTUARIO.isin(set(cal.ID_PRONTUARIO))]    return (tr, cal) if len(tr) > 200 else (None, None)def dobras_temporais(d, k=3):    """Blocos temporais para validação cruzada interna."""    meses = np.sort(d.MES.astype(str).unique())    if len(meses) < k + 1:        return None    ordem = pd.Series(d.MES.astype(str)).map({m: i for i, m in enumerate(meses)}).values    pront, y = d.ID_PRONTUARIO.values, d.Y.values    out = []    for bloco in np.array_split(np.arange(len(meses)), k)[1:]:        iva = np.where(np.isin(ordem, bloco))[0]        itr = np.where(ordem < bloco[0])[0]        if len(iva) and len(itr):            itr = itr[~np.isin(pront[itr], set(pront[iva]))]        if len(iva) > 30 and len(itr) > 100 and len(np.unique(y[iva])) > 1:            out.append((itr, iva))    return out or None

## 5. O achado de método mais reutilizável### Parametrização fixa pode fabricar uma conclusãoNa primeira versão da análise, o modelo local da menor unidade produziu AUROC de**0,500 exato** — predição constante. A leitura natural seria "com 39 eventos nãohá sinal suficiente". **Estava errada.**O XGBoost estima `base_score` a partir da taxa base (0,0042 nesta coorte). Ohessiano por linha vale então `p(1−p) ≈ 0,0042`, e exigir `min_child_weight = 20`equivale a exigir **4.738 linhas por folha** — em um treino de 9.199. Nenhumadivisão satisfaz a restrição, e o modelo vira constante.| `min_child_weight` | árvores com divisão | AUROC ||---|---|---|| 20 | **0** | **0,500** || 10 | 300 | **0,717** || 5 | 300 | 0,685 || 1 | 300 | 0,657 |O mesmo dado, com floresta aleatória: **0,727**.### Por que isso importa para a literatura de transferênciaHiperparâmetros fixos penalizam mais o braço com menos dados. Como o braço localé exatamente o que tem menos dados nas unidades pequenas, a penalização é**correlacionada com o porte da unidade** — e produz, sozinha, uma aparente leide que "a transferência ajuda mais onde há menos dado".Com ajuste por unidade, nesta coorte:| | fixo | ajustado ||---|---|---|| local | 0,7059 | **0,7400** || agrupado | 0,7411 | 0,7529 || ganho da agregação | +0,0352 | **+0,0129** || ρ(eventos, ganho) | −0,743 (p = 0,002) | **−0,114 (p = 0,685)** |A correlação desaparece. **A lei era artefato da parametrização.**O mesmo vale para a continuação de boosting: com parâmetros fixos ela rendia0,6886 e o número ótimo de árvores parecia ser 2; ajustada, rende **0,7520** commediana de **49 árvores**.> **Recomendação.** Em comparações entre estratégias de transferência, ajuste os> hiperparâmetros **por unidade e por estratégia**, e verifique se algum modelo> colapsou (zero divisões) antes de interpretar um AUROC próximo de 0,5.

In [ ]:
def divisoes(modelo):    """Quantas árvores efetivamente se ramificaram. Zero = modelo constante."""    return sum(1 for t in modelo.get_booster().get_dump() if "[" in t)def diagnostico(modelo, y, p):    d = divisoes(modelo)    if d == 0:        raise RuntimeError(            "modelo constante: nenhuma árvore se ramificou. "            "Verifique min_child_weight contra a taxa base do desfecho.")    if abs(roc_auc_score(y, p) - .5) < 1e-6:        raise RuntimeError("AUROC exatamente 0,5: predição degenerada.")    return d

## 6. Estratégias comparadas| estratégia | procedimento | atravessa a fronteira ||---|---|---|| local | treina só no destino | nada || externa + Platt | modelo da origem, recalibrado no destino | parâmetros || agrupada | treina sobre origem e destino | **dado de paciente** || continuação | acrescenta árvores ao modelo da origem | parâmetros |A recalibração de Platt é ajustada na partição de 2024 reservada, nunca no teste.Por ser monotônica, **não altera a AUROC**: corrige a escala sem tocar naordenação.

In [ ]:
def logit(p):    p = np.clip(p, 1e-6, 1 - 1e-6)    return np.log(p / (1 - p))def recalibra(p_cal, y_cal, p_novo):    m = LogisticRegression(max_iter=1000).fit(logit(p_cal).reshape(-1, 1), y_cal)    return m.predict_proba(logit(p_novo).reshape(-1, 1))[:, 1]def continuacao(caminho_origem, X_tr, y_tr, X_te, n_arvores, params):    """Transferência por parâmetro: acrescenta árvores ao modelo da origem."""    import xgboost as xgb    b = xgb.Booster(); b.load_model(caminho_origem); b.set_param(params)    n0 = b.num_boosted_rounds()    dtr = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)    for i in range(n_arvores):        b.update(dtr, iteration=n0 + i)    return b.predict(xgb.DMatrix(X_te, enable_categorical=True))

## 7. MétricasAUROC responde se o modelo ordena. Nenhuma decisão clínica se toma com isso.Sob prevalência de 0,64%, reportam-se também:- **precisão média**, mais informativa que a AUROC nesse regime;- **razão observado/esperado (O:E)** e escore de Brier, para calibração;- **violação de multicalibração**, α = máximo de |log O:E| sobre subgrupos;- **benefício líquido**, que responde se o modelo supera não fazer nada.> α é um **máximo** sobre subgrupos e sofre de multiplicidade: parte do valor> pode ser ruído do grupo mais extremo. Interpretar com o número de subgrupos> avaliados à vista.

In [ ]:
def metricas(y, p):    k = max(1, int(len(p) * .05))    topo = np.argsort(-p)[:k]    esp = float(p.mean())    oe = float(y.mean() / esp) if esp > 0 else np.nan    return {"auroc": roc_auc_score(y, p),            "precisao_media": average_precision_score(y, p),            "brier": brier_score_loss(y, p),            "oe": oe,            "abs_log_oe": abs(np.log(oe)) if oe and oe > 0 else np.nan,            "lift5": float(y[topo].mean() / y.mean()) if y.mean() > 0 else np.nan}def beneficio_liquido(y, p, pt):    """Vickers & Elkin (2006). pt = risco a partir do qual vale agir."""    alarma = p >= pt    n = len(y)    tp = int((alarma & (y == 1)).sum()); fp = int((alarma & (y == 0)).sum())    return tp / n - (fp / n) * (pt / (1 - pt))def alpha_multicalibracao(y, p, grupos, min_n=300, min_ev=5):    """grupos: dict nome -> máscara booleana. Retorna (alpha, pior grupo, n)."""    pior, nome, avaliados = 0.0, None, 0    for k, m in grupos.items():        if m.sum() < min_n or y[m].sum() < min_ev:            continue        avaliados += 1        esp = p[m].mean()        if esp <= 0:            continue        o = y[m].mean() / esp        if o > 0 and abs(np.log(o)) > pior:            pior, nome = abs(np.log(o)), k    return pior, nome, avaliados

## 8. Correção de priori para contexto enriquecidoModelos fundacionais tabulares têm limite prático de contexto. Com prevalência de0,64%, um contexto sorteado ao acaso conteria pouquíssimos positivos, entãopreservam-se todos os positivos e subamostram-se os negativos. **Isso não éreamostragem sintética**: nenhuma linha é duplicada.Mas desloca a prevalência do contexto, e a razão O:E resultante fica em torno de0,10 — o que pareceria defeito grave do método. Não é. Amostragem caso-controledesloca o logito por constante **conhecida do desenho amostral**: mantida afração *f* dos negativos, soma-se log(*f*) ao logito.Nesta coorte, a correção leva o O:E de ~0,10 para **0,94–1,02** e o erro absolutodo logaritmo de 1,67 para **0,14**, sem alterar a AUROC.

In [ ]:
def corrige_priori(p, fracao_negativos_preservada):    """Correção exata para contexto com negativos subamostrados."""    f = float(np.clip(fracao_negativos_preservada, 1e-6, 1.0))    return 1 / (1 + np.exp(-(logit(p) + np.log(f))))

## 9. O que os resultados mostraramCom hiperparâmetros ajustados por unidade e por estratégia, em 15 unidades depronto socorro (treino 2024, teste 2025):| estratégia | AUROC | vence em ||---|---|---|| agrupada | 0,7529 | 7/15 || continuação de boosting | 0,7520 | 5/15 || local | 0,7400 | 3/15 || externa + Platt | 0,7156 | 0/15 |**A agregação e a continuação empatam** (diferença de 0,0009), mas apenas asegunda dispensa o compartilhamento de microdado de paciente.Sobre calibração, a estratégia de **melhor calibração global** apresentou a**pior calibração por subgrupo**, subestimando o risco de homens com 75 anos oumais por fator de 1,9. Recalibração global dispõe de um único par de parâmetros enão corrige gradiente etário.Sobre utilidade clínica, no ponto de operação de 2% das altas capturam-se ~15%dos eventos, com 24 pacientes avaliados por evento identificado. O modelo superatratar todos e não tratar ninguém em toda a faixa de limiar entre 0,2% e 5%.## 10. Resultados negativosReportados porque poupam esforço a quem seguir este caminho:- **Orientar o desenvolvimento ao benefício líquido não compensa.** Peso  assimétrico, recalibração restrita à região de decisão e seleção da estratégia  por benefício líquido, nenhum superou o modelo padrão bem calibrado.- **Ajuste fino de modelo fundacional rendeu +0,004 a +0,006**, dentro da  variação amostral. O aprendizado em contexto já extrai o disponível.- **Pós-processamento multicalibrado estimado localmente piora.** Com ~70 eventos  na partição de calibração, um subgrupo tem ~10; ajustar dezenas de correções  sobre isso é ajustar ruído. A versão federada, somando as demais unidades,  melhora em 10 de 15, mas não garante: a rede inteira reúne 90 eventos no  subgrupo de homens com 75 anos ou mais.## Licença e citaçãoCódigo sob licença MIT. Os dados **não** são distribuídos.Se este material for útil, cite o trabalho associado.